# MLFlow Experiment Tracking

Objective:
Track machine learning experiments using MLFlow
and publish model runs to DagsHub.

In [18]:
import pandas as pd
import numpy as np

import mlflow
import dagshub

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

print("Libraries loaded successfully")

Libraries loaded successfully


In [19]:
dagshub.init(
    repo_owner="snp-007",
    repo_name="energy-drink-price-prediction",
    mlflow=True
)

Initialized MLflow to track repo "snp-007/energy-drink-price-prediction"

Repository snp-007/energy-drink-price-prediction initialized!

In [20]:
mlflow.set_experiment(
    "Beverage Price Prediction"
)

<Experiment: artifact_location='mlflow-artifacts:/3035e2e56b1e4987899fd1c49a286f1b', creation_time=1779951756265, experiment_id='0', last_update_time=1779951756265, lifecycle_stage='active', name='Beverage Price Prediction', tags={}, trace_location=None, workspace='default'>

In [21]:
mlflow.get_tracking_uri()

'https://dagshub.com/snp-007/energy-drink-price-prediction.mlflow'

In [32]:
import joblib

In [26]:
df = pd.read_csv(
    "../data/processed/survey_results_feature_engineered.csv"
)

print(df.shape)

(29965, 20)


In [25]:
from sklearn.model_selection import train_test_split

from sklearn.preprocessing import (
    LabelEncoder,
    StandardScaler
)

In [27]:
df.drop(columns='respondent_id', inplace=True)

X = df.drop(columns='price_range')

y = df['price_range']

In [28]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42
)

In [29]:
label_encode_cols = [
    'age_group',
    'income_levels',
    'health_concerns',
    'consume_frequency(weekly)',
    'preferable_consumption_size'
]

In [30]:
feature_label_encoders = {}

for col in label_encode_cols:

    le = LabelEncoder()

    X_train[col] = le.fit_transform(X_train[col])

    X_test[col] = le.transform(X_test[col])

    feature_label_encoders[col] = le

In [31]:
remaining_cat_cols = X_train.select_dtypes(
    include='object'
).columns.tolist()

X_train = pd.get_dummies(
    X_train,
    columns=remaining_cat_cols,
    drop_first=True
)

X_test = pd.get_dummies(
    X_test,
    columns=remaining_cat_cols,
    drop_first=True
)

X_train, X_test = X_train.align(
    X_test,
    join='left',
    axis=1,
    fill_value=0
)

In [33]:
target_encoder = joblib.load(
    "../outputs/models/target_encoder.pkl"
)

y_train_encoded = target_encoder.transform(y_train)

y_test_encoded = target_encoder.transform(y_test)

In [35]:
scaler = joblib.load(
    "../outputs/models/scaler.pkl"
)

X_train_scaled = scaler.transform(X_train)

X_test_scaled = scaler.transform(X_test)

# MLFlow Tracking - Gaussian Naive Bayes

This section logs the Gaussian Naive Bayes model
to MLFlow and publishes the run to DagsHub.

In [ ]:
import mlflow.sklearn

In [23]:
gnb_model = joblib.load(
    "../outputs/models/gnb_model.pkl"
)

print("Gaussian NB model loaded successfully")

Gaussian NB model loaded successfully


In [36]:
with mlflow.start_run(run_name="Gaussian Naive Bayes"):

    # Train model
    gnb_model.fit(
        X_train_scaled,
        y_train_encoded
    )

    # Predictions
    gnb_predictions = gnb_model.predict(
        X_test_scaled
    )

    # Metrics
    accuracy = accuracy_score(
        y_test_encoded,
        gnb_predictions
    )

    precision = precision_score(
        y_test_encoded,
        gnb_predictions,
        average='weighted'
    )

    recall = recall_score(
        y_test_encoded,
        gnb_predictions,
        average='weighted'
    )

    f1 = f1_score(
        y_test_encoded,
        gnb_predictions,
        average='weighted'
    )

    # Log parameters
    mlflow.log_param(
        "model_type",
        "GaussianNB"
    )

    # Log metrics
    mlflow.log_metric(
        "accuracy",
        accuracy
    )

    mlflow.log_metric(
        "precision",
        precision
    )

    mlflow.log_metric(
        "recall",
        recall
    )

    mlflow.log_metric(
        "f1_score",
        f1
    )

    # Log model
    mlflow.sklearn.log_model(
        gnb_model,
        "model"
    )

    print("Gaussian NB logged successfully")

2026/05/28 12:44:34 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/28 12:44:37 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Gaussian NB logged successfully
🏃 View run Gaussian Naive Bayes at: https://dagshub.com/snp-007/energy-drink-price-prediction.mlflow/#/experiments/0/runs/1a516c7caa8e43518e451a5c5e649ad6
🧪 View experiment at: https://dagshub.com/snp-007/energy-drink-price-prediction.mlflow/#/experiments/0


# Reusable MLFlow Logging Pipeline

In [37]:
lr_model = joblib.load(
    "../outputs/models/lr_model.pkl"
)

svm_model = joblib.load(
    "../outputs/models/svm_model.pkl"
)

rf_model = joblib.load(
    "../outputs/models/rf_model.pkl"
)

xgb_model = joblib.load(
    "../outputs/models/xgb_model.pkl"
)

lgbm_model = joblib.load(
    "../outputs/models/best_model_lightgbm.pkl"
)

print("All models loaded successfully")

All models loaded successfully


In [38]:
def log_model_to_mlflow(
    model_name,
    model,
    X_test_data,
    y_test_data
):

    with mlflow.start_run(run_name=model_name):

        # Predictions
        predictions = model.predict(
            X_test_data
        )

        # Metrics
        accuracy = accuracy_score(
            y_test_data,
            predictions
        )

        precision = precision_score(
            y_test_data,
            predictions,
            average='weighted'
        )

        recall = recall_score(
            y_test_data,
            predictions,
            average='weighted'
        )

        f1 = f1_score(
            y_test_data,
            predictions,
            average='weighted'
        )

        # Parameters
        mlflow.log_param(
            "model_type",
            model.__class__.__name__
        )

        # Metrics
        mlflow.log_metric(
            "accuracy",
            accuracy
        )

        mlflow.log_metric(
            "precision",
            precision
        )

        mlflow.log_metric(
            "recall",
            recall
        )

        mlflow.log_metric(
            "f1_score",
            f1
        )

        # Model Artifact
        mlflow.sklearn.log_model(
            sk_model=model,
            name="model"
        )

        print(f"{model_name} logged successfully")

In [39]:
log_model_to_mlflow(
    model_name="Logistic Regression",
    model=lr_model,
    X_test_data=X_test_scaled,
    y_test_data=y_test_encoded
)

2026/05/28 12:48:27 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Logistic Regression logged successfully
🏃 View run Logistic Regression at: https://dagshub.com/snp-007/energy-drink-price-prediction.mlflow/#/experiments/0/runs/4ad68943884a41f99910f6fd6bd55e2e
🧪 View experiment at: https://dagshub.com/snp-007/energy-drink-price-prediction.mlflow/#/experiments/0


In [40]:
log_model_to_mlflow(
    model_name="Support Vector Machine",
    model=svm_model,
    X_test_data=X_test_scaled,
    y_test_data=y_test_encoded
)

2026/05/28 12:49:11 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Support Vector Machine logged successfully
🏃 View run Support Vector Machine at: https://dagshub.com/snp-007/energy-drink-price-prediction.mlflow/#/experiments/0/runs/6d24bb4f3b5f4d6da15f2bf7f29ddb93
🧪 View experiment at: https://dagshub.com/snp-007/energy-drink-price-prediction.mlflow/#/experiments/0


In [41]:
log_model_to_mlflow(
    model_name="Random Forest",
    model=rf_model,
    X_test_data=X_test,
    y_test_data=y_test_encoded
)

2026/05/28 12:49:32 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Random Forest logged successfully
🏃 View run Random Forest at: https://dagshub.com/snp-007/energy-drink-price-prediction.mlflow/#/experiments/0/runs/42caabf8fbd649d989b114f526f0dec1
🧪 View experiment at: https://dagshub.com/snp-007/energy-drink-price-prediction.mlflow/#/experiments/0


In [42]:
log_model_to_mlflow(
    model_name="XGBoost",
    model=xgb_model,
    X_test_data=X_test,
    y_test_data=y_test_encoded
)

2026/05/28 12:52:42 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


XGBoost logged successfully
🏃 View run XGBoost at: https://dagshub.com/snp-007/energy-drink-price-prediction.mlflow/#/experiments/0/runs/40b3f57a6c954fc6bb0200ec7c4921f7
🧪 View experiment at: https://dagshub.com/snp-007/energy-drink-price-prediction.mlflow/#/experiments/0


In [43]:
log_model_to_mlflow(
    model_name="LGBMClassifier",
    model=lgbm_model,
    X_test_data=X_test,
    y_test_data=y_test_encoded
)


2026/05/28 12:53:07 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


LGBMClassifier logged successfully
🏃 View run LGBMClassifier at: https://dagshub.com/snp-007/energy-drink-price-prediction.mlflow/#/experiments/0/runs/f8f009d954ad4b3fb2e67adf5bd14e70
🧪 View experiment at: https://dagshub.com/snp-007/energy-drink-price-prediction.mlflow/#/experiments/0
